# 5. Model Preparation

## Objective

This notebook prepares the feature-engineered dataset for machine learning.

The main steps include:

- Separating input features and the target variable.
- Splitting the dataset into training, validation, and test sets.
- Preserving the target class distribution using stratified sampling.
- Separating categorical and numerical features.
- Building a preprocessing pipeline using Scikit-learn.
- Preventing data leakage by fitting preprocessing steps only on the training data.

No machine learning model is trained in this notebook.

In [38]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [39]:
df = pd.read_csv(
    "../data/processed/credit_default_features.csv"
)

df.head()

,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,RECENT_DELAY,AVG_BILL_AMT,MAX_BILL_AMT,AVG_PAY_AMT,TOTAL_PAY_AMT,ZERO_PAYMENT_MONTHS,CREDIT_UTILIZATION,AVG_CREDIT_UTILIZATION,PAYMENT_TO_BILL_RATIO,BILL_CHANGE
0,20000,2,2,1,24,2,2,-1,-1,-2,...,1,1284.000000,3913,114.833333,689,5,0.195650,0.064200,0.089434,3913
1,120000,2,2,2,26,-1,2,0,0,0,...,0,2846.166667,3455,833.333333,5000,2,0.022350,0.023718,0.292791,-579
2,90000,2,2,2,34,0,0,0,0,0,...,0,16942.166667,29239,1836.333333,11018,0,0.324878,0.188246,0.108388,13690
3,50000,2,2,1,37,0,0,0,0,0,...,0,38555.666667,49291,1398.000000,8388,0,0.939800,0.771113,0.036259,17443
4,50000,1,2,1,57,-1,0,-1,0,0,...,0,18223.166667,35835,9841.500000,59049,0,0.172340,0.364463,0.540054,-10514


In [40]:
df.shape

(30000, 37)

In [41]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 37 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   LIMIT_BAL                   30000 non-null  int64  
 1   SEX                         30000 non-null  int64  
 2   EDUCATION                   30000 non-null  int64  
 3   MARRIAGE                    30000 non-null  int64  
 4   AGE                         30000 non-null  int64  
 5   PAY_0                       30000 non-null  int64  
 6   PAY_2                       30000 non-null  int64  
 7   PAY_3                       30000 non-null  int64  
 8   PAY_4                       30000 non-null  int64  
 9   PAY_5                       30000 non-null  int64  
 10  PAY_6                       30000 non-null  int64  
 11  BILL_AMT1                   30000 non-null  int64  
 12  BILL_AMT2                   30000 non-null  int64  
 13  BILL_AMT3                   30000 non-null

In [42]:
X = df.drop(columns=["DEFAULT_PAYMENT_NEXT_MONTH"])
y = df["DEFAULT_PAYMENT_NEXT_MONTH"]

In [43]:
X.shape, y.shape

((30000, 36), (30000,))

In [44]:
y.head()

0    1
1    1
2    0
3    0
4    0
Name: DEFAULT_PAYMENT_NEXT_MONTH, dtype: int64

In [45]:
y.value_counts()

DEFAULT_PAYMENT_NEXT_MONTH
0    23364
1     6636
Name: count, dtype: int64

In [46]:
y.value_counts(normalize=True).mul(100).round(2)

DEFAULT_PAYMENT_NEXT_MONTH
0    77.88
1    22.12
Name: proportion, dtype: float64

In [47]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [48]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [49]:
print("Training set:")
print(X_train.shape, y_train.shape)

print("\nValidation set:")
print(X_val.shape, y_val.shape)

print("\nTest set:")
print(X_test.shape, y_test.shape)

Training set:
(21000, 36) (21000,)

Validation set:
(4500, 36) (4500,)

Test set:
(4500, 36) (4500,)


In [50]:
def show_class_distribution(name, target):
    distribution = (
        target.value_counts(normalize=True)
        .sort_index()
        .mul(100)
        .round(2)
    )

    print(f"{name}:")
    print(distribution)
    print()

In [51]:
show_class_distribution("Full dataset", y)
show_class_distribution("Training set", y_train)
show_class_distribution("Validation set", y_val)
show_class_distribution("Test set", y_test)

Full dataset:
DEFAULT_PAYMENT_NEXT_MONTH
0    77.88
1    22.12
Name: proportion, dtype: float64

Training set:
DEFAULT_PAYMENT_NEXT_MONTH
0    77.88
1    22.12
Name: proportion, dtype: float64

Validation set:
DEFAULT_PAYMENT_NEXT_MONTH
0    77.89
1    22.11
Name: proportion, dtype: float64

Test set:
DEFAULT_PAYMENT_NEXT_MONTH
0    77.87
1    22.13
Name: proportion, dtype: float64



In [52]:
categorical_features = [
    "SEX",
    "EDUCATION",
    "MARRIAGE"
]

In [53]:
binary_features = [
    "HAS_PAYMENT_DELAY",
    "RECENT_DELAY"
]

In [54]:
numerical_features = [
    col for col in X.columns
    if col not in categorical_features
]

In [55]:
print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

print("\nNumber of categorical features:", len(categorical_features))
print("Number of numerical features:", len(numerical_features))

Categorical features:
['SEX', 'EDUCATION', 'MARRIAGE']

Numerical features:
['LIMIT_BAL', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'NUM_DELAYED_MONTHS', 'MAX_PAYMENT_DELAY', 'HAS_PAYMENT_DELAY', 'RECENT_DELAY', 'AVG_BILL_AMT', 'MAX_BILL_AMT', 'AVG_PAY_AMT', 'TOTAL_PAY_AMT', 'ZERO_PAYMENT_MONTHS', 'CREDIT_UTILIZATION', 'AVG_CREDIT_UTILIZATION', 'PAYMENT_TO_BILL_RATIO', 'BILL_CHANGE']

Number of categorical features: 3
Number of numerical features: 33


In [56]:
numerical_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

In [57]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [58]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numerical_pipeline,
            numerical_features
        ),
        (
            "cat",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [59]:
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

In [60]:
X_train_processed = preprocessor.transform(X_train)

X_val_processed = preprocessor.transform(X_val)

X_test_processed = preprocessor.transform(X_test)

In [61]:
print(
    "Train:",
    X_train.shape,
    "→",
    X_train_processed.shape
)

print(
    "Validation:",
    X_val.shape,
    "→",
    X_val_processed.shape
)

print(
    "Test:",
    X_test.shape,
    "→",
    X_test_processed.shape
)

Train: (21000, 36) → (21000, 42)
Validation: (4500, 36) → (4500, 42)
Test: (4500, 36) → (4500, 42)


In [62]:
feature_names = (
    preprocessor
    .get_feature_names_out()
)

feature_names[:20]

array(['num__LIMIT_BAL', 'num__AGE', 'num__PAY_0', 'num__PAY_2',
       'num__PAY_3', 'num__PAY_4', 'num__PAY_5', 'num__PAY_6',
       'num__BILL_AMT1', 'num__BILL_AMT2', 'num__BILL_AMT3',
       'num__BILL_AMT4', 'num__BILL_AMT5', 'num__BILL_AMT6',
       'num__PAY_AMT1', 'num__PAY_AMT2', 'num__PAY_AMT3', 'num__PAY_AMT4',
       'num__PAY_AMT5', 'num__PAY_AMT6'], dtype=object)

In [63]:
assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(y_test)

assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)

assert "DEFAULT" not in X_train.columns

print("Train/validation/test split checks passed.")

Train/validation/test split checks passed.


## Model Preparation Summary

The feature-engineered dataset was successfully prepared for machine learning.

### Data Split

The dataset was divided into:

- 70% Training set
- 15% Validation set
- 15% Test set

Stratified sampling was used to preserve the original class distribution.

### Preprocessing

Categorical features:

- `SEX`
- `EDUCATION`
- `MARRIAGE`

were encoded using One-Hot Encoding.

Numerical and ordinal features were standardized using `StandardScaler`.

### Data Leakage Prevention

The preprocessing pipeline was fitted exclusively on the training data.

Validation and test data were transformed using preprocessing parameters
learned from the training set.

The test set will remain untouched until final model evaluation.